# Synthesis Agent — Tutorial Walkthrough

This notebook walks through the **Synthesis Agent** logic step by step:

1. **What it does**: Takes a paper (title + abstract) and extracts structured fields using Claude
2. **System prompt design**: How we instruct the LLM to extract wearables-aware metadata
3. **Pydantic validation**: How we enforce schema compliance on LLM output
4. **Prompt caching**: How we save cost when processing many papers
5. **Retry logic**: How we handle malformed LLM responses

---

In [ ]:
import sys, json
sys.path.insert(0, "..")

from lit_review_agent.state import Paper
from lit_review_agent.synthesis import SynthesisExtraction, _SYSTEM_PROMPT
print("Imports OK")

## 1. The Input: A `Paper` Object

The search tools (Phase 1) return `Paper` objects with metadata from PubMed or Semantic Scholar.
At this point, only the **bibliographic fields** are filled — the synthesis fields are all `None` / `[]`.

In [ ]:
# A sample paper (like what search_pubmed returns)
sample_paper = Paper(
    source="pubmed",
    pmid="38123456",
    doi="10.1234/example.2024",
    title="Deep Learning for Atrial Fibrillation Detection Using Consumer Smartwatch PPG Signals",
    authors=["Smith J", "Doe A", "Lee K"],
    year=2024,
    abstract=(
        "Background: Atrial fibrillation (AFib) is the most common cardiac arrhythmia. "
        "Consumer smartwatches with photoplethysmography (PPG) sensors offer a scalable "
        "screening approach. Methods: We developed a 1D convolutional neural network trained "
        "on PPG recordings from 2,450 participants (ages 35-85, 48% female, 22% Black) "
        "wearing Apple Watch Series 7. Ground truth was established via simultaneous 12-lead "
        "ECG. Data were split at the participant level using 5-fold cross-validation with a "
        "held-out external validation cohort (N=500). Results: The model achieved sensitivity "
        "of 94.2%, specificity of 97.1%, and AUROC of 0.982. Performance was lower in the "
        "65+ age subgroup (sensitivity 89.3%). Conclusions: Deep learning on consumer PPG "
        "shows strong AFib detection performance but age-related performance gaps warrant "
        "further investigation."
    ),
)

print("=== Paper BEFORE synthesis ===")
print(f"Title: {sample_paper.title}")
print(f"Task: {sample_paper.task}")         # None
print(f"Devices: {sample_paper.devices}")   # []
print(f"Metrics: {sample_paper.evaluation_metrics}")  # []

## 2. The System Prompt

The Synthesis Agent uses a carefully designed system prompt that:
- Defines the **role** (biomedical research synthesis assistant)
- Specifies the **exact JSON schema** to return
- Includes **rules** to prevent hallucination (extract ONLY from abstract)
- Is **cacheable** — same prompt reused across all papers, saving ~90% on input tokens

Let's look at it:

In [ ]:
print(_SYSTEM_PROMPT)

## 3. The User Message

For each paper, we construct a simple user message with the paper's metadata.
This is the **variable** part — it changes per paper.

In [ ]:
# This is what gets sent as the user message to Claude
user_content = (
    f"Title: {sample_paper.title}\n"
    f"Authors: {', '.join(sample_paper.authors[:10])}\n"
    f"Year: {sample_paper.year}\n"
    f"Abstract: {sample_paper.abstract}"
)

print(user_content)

## 4. Pydantic Validation

Claude returns raw JSON text. We parse it and validate against `SynthesisExtraction` — a Pydantic model that mirrors the synthesis fields of `Paper`.

If the JSON is malformed or doesn't match the schema, Pydantic raises a `ValidationError` and we **retry**.

In [ ]:
from pydantic import ValidationError

# Simulate a GOOD response from Claude
good_json = '''{
  "task": "AFib detection",
  "devices": ["Apple Watch Series 7"],
  "sensor_modalities": ["PPG"],
  "cohort_size": 2450,
  "cohort_demographics": "ages 35-85, 48% female, 22% Black",
  "reference_standard": "12-lead ECG",
  "split_strategy": "participant-level 5-fold CV with external validation (N=500)",
  "evaluation_metrics": ["sensitivity", "specificity", "AUROC"],
  "key_findings": "Model achieved 94.2% sensitivity, 97.1% specificity, AUROC 0.982; lower performance in 65+ age subgroup",
  "limitations": "Age-related performance gaps; single device type",
  "quality_notes": "Strong design: participant-level splits, external validation, demographic subgroup analysis"
}'''

try:
    extraction = SynthesisExtraction(**json.loads(good_json))
    print("Validation PASSED")
    print(f"  task = {extraction.task}")
    print(f"  devices = {extraction.devices}")
    print(f"  cohort_size = {extraction.cohort_size}")
    print(f"  evaluation_metrics = {extraction.evaluation_metrics}")
except ValidationError as e:
    print(f"Validation FAILED: {e}")

In [ ]:
# Simulate a BAD response — e.g. Claude wraps in markdown or returns wrong types
bad_json = '''```json
{"task": "AFib detection", "cohort_size": "two thousand"}
```'''

# Step 1: strip markdown fences (the agent does this automatically)
raw = bad_json.strip()
if raw.startswith("```"):
    raw = raw.strip("`").removeprefix("json").strip()

print(f"After fence stripping: {raw[:80]}...")

# Step 2: try to parse + validate
try:
    extraction = SynthesisExtraction(**json.loads(raw))
    print(f"Validation passed (might be partial): task={extraction.task}")
except (json.JSONDecodeError, ValidationError) as e:
    print(f"\nValidation FAILED (would retry): {type(e).__name__}")
    print(f"  {e}")

## 5. Merging Extraction into Paper

Once validated, we merge the extracted fields into a **copy** of the original Paper.
This preserves all bibliographic metadata while adding the synthesis results.

In [ ]:
# Merge extraction into the paper (same as what synthesize_paper does)
extraction = SynthesisExtraction(**json.loads(good_json))
updated = sample_paper.model_copy(update=extraction.model_dump(exclude_none=True))

print("=== Paper AFTER synthesis ===")
print(f"Title: {updated.title}")        # preserved from original
print(f"PMID: {updated.pmid}")           # preserved
print(f"Task: {updated.task}")           # NEW: from extraction
print(f"Devices: {updated.devices}")     # NEW
print(f"Cohort size: {updated.cohort_size}")  # NEW
print(f"Metrics: {updated.evaluation_metrics}")  # NEW
print(f"Key findings: {updated.key_findings}")  # NEW

## 6. Prompt Caching — How It Saves Money

The system prompt is the same for every paper. Anthropic's **prompt caching** lets us mark it as `cache_control: ephemeral`, so after the first call:

- **First paper**: Full system prompt tokens billed at normal rate (`cache_creation_input_tokens`)
- **Subsequent papers**: Cached tokens billed at ~10% of normal rate (`cache_read_input_tokens`)

In the API call, we pass the system prompt like this:

```python
system=[
    {
        "type": "text",
        "text": _SYSTEM_PROMPT,
        "cache_control": {"type": "ephemeral"},  # <-- enables caching
    }
]
```

The synthesis function logs cache stats for each call so you can verify it's working:
```
Synthesis [12345678] attempt=0 input=800 cache_read=600 cache_create=0 output=200
                                         ^^^^^^^^^^
                                         This > 0 means caching is active!
```

## 7. Retry Logic

The synthesis function retries up to `max_retries` times (default: 2) when:
- Claude returns invalid JSON
- The JSON doesn't match the Pydantic schema

If all retries are exhausted, it returns the **original paper unmodified** (no synthesis fields filled) rather than crashing. This ensures one bad paper doesn't stop the entire pipeline.

```
Synthesis parse error (attempt 1/3): Expecting value: line 1 column 1
Synthesis parse error (attempt 2/3): Expecting value: line 1 column 1
Synthesis parse error (attempt 3/3): Expecting value: line 1 column 1
Synthesis failed after 3 retries for: <paper title>
```

## 8. Run it for real (requires API key)

Uncomment and run the cell below to test against the live Anthropic API.

In [ ]:
# Uncomment to run against the live API:

# import logging
# logging.basicConfig(level=logging.INFO)
# from lit_review_agent.synthesis import synthesize_paper
#
# result = synthesize_paper(sample_paper)
# print(result.model_dump_json(indent=2))